# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [2]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [5]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`
sample_embedding = embeddings.embed_query("What is the capital of France?")

👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [6]:
print(type(sample_embedding))   # list
print(len(sample_embedding))    # embedding dimension (3072 for gemini-embedding-001)
sample_embedding[:5]

<class 'list'>
3072


[-0.032554302, 0.01305496, 0.015067407, -0.07128718, -0.031062366]

## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/CRE-10-2026-10-05_EN.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [8]:
print(type(pages))                                   # list
print(len(pages))                                    # 113
print(type(pages[0]))                                # Document
print(pages[30].page_content[:500])                  # content of one page
print(sum(len(p.page_content) for p in pages))      # total characters
pages[30].metadata

<class 'list'>
113
<class 'langchain_core.documents.base.Document'>
05-10-2026  31 
lycéenne, mais bien davantage le drapeau palestinien. C'est pourquoi cette assemblée doit 
s'emparer de ce sujet selon les termes proposés par notre groupe. 
 
1-0024-0000 
Emma Rafowicz, au nom du groupe S&D. – Madame la Présidente, mes chers collègues, les lycéens 
se révoltent partout en France. Ils le font car leurs conditions d'études sont insupportables: 
délabrement des établissements scolaires, câbles électriques qui pendent, plafonds crevés, toilettes 
dans un état lamen
329034


{'producer': 'Aspose.Words for Java 24.2.0',
 'creator': 'Aspose.Words',
 'creationdate': '',
 'author': 'e-Parliament@europarl.europa.eu',
 'dmxml.render.id': '652525',
 'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa',
 'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml',
 'source': 'data/CRE-10-2026-10-05_EN.pdf',
 'total_pages': 113,
 'page': 30,
 'page_label': '31'}

## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [9]:
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

329258

Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
all_splits = text_splitter.split_documents(pdf)

👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [11]:
print(type(all_splits))
print(len(all_splits))
print(type(all_splits[0]))
print(all_splits[0].page_content[:500])
print(sum(len(s.page_content) for s in all_splits))  # more than pdf_text because of the overlap
all_splits[0].metadata

<class 'list'>
207
<class 'langchain_core.documents.base.Document'>
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN
402537


{'producer': 'Aspose.Words for Java 24.2.0',
 'creator': 'Aspose.Words',
 'creationdate': '',
 'author': 'e-Parliament@europarl.europa.eu',
 'dmxml.render.id': '652525',
 'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa',
 'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml',
 'source': 'data/CRE-10-2026-10-05_EN.pdf',
 'total_pages': 113}

## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [14]:
# Import the necessary libraries
import time
from langchain_core.vectorstores import InMemoryVectorStore

def add_in_batches(vector_store, docs, batch_size=10, pause=15):
    """Add documents in small batches to stay under the free-tier tokens-per-minute limit."""
    ids = []
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        while True:
            try:
                ids += vector_store.add_documents(documents=batch)
                break
            except Exception as e:
                if "429" in str(e):
                    print(f"Quota hit at chunk {i}, waiting 60s...")
                    time.sleep(60)
                else:
                    raise
        print(f"{min(i + batch_size, len(docs))}/{len(docs)} chunks embedded")
        time.sleep(pause)
    return ids

# Create an in-memory vector store using the embedder `embeddings` we created earlier
vector_store = InMemoryVectorStore(embeddings)

# Add the `all_splits` to the vector store and store the result in a variable called `document_ids`
document_ids = add_in_batches(vector_store, all_splits)

10/207 chunks embedded
20/207 chunks embedded
30/207 chunks embedded
40/207 chunks embedded
50/207 chunks embedded
60/207 chunks embedded
70/207 chunks embedded
80/207 chunks embedded
90/207 chunks embedded
100/207 chunks embedded
110/207 chunks embedded
120/207 chunks embedded
130/207 chunks embedded
140/207 chunks embedded
150/207 chunks embedded
160/207 chunks embedded
170/207 chunks embedded
180/207 chunks embedded
190/207 chunks embedded
200/207 chunks embedded
207/207 chunks embedded


In [15]:
# Have a look at the first 3 document IDs
document_ids[:3]

['20e269ef-3196-4ac9-b045-db58434bd8a7',
 '980f6555-1a01-4744-9a48-cd77bd90909c',
 '229f5aab-2c58-44e7-a4f0-8e89b475c24f']

In [16]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.
docs = vector_store.get_by_ids(document_ids[:3])
docs

[Document(id='20e269ef-3196-4ac9-b045-db58434bd8a7', metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '652525', 'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa', 'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-10-05_EN.pdf', 'total_pages': 113}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nT

👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [17]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.
docs = vector_store.get_by_ids(document_ids[:3])
docs

[Document(id='20e269ef-3196-4ac9-b045-db58434bd8a7', metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '652525', 'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa', 'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-10-05_EN.pdf', 'total_pages': 113}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nT

In [18]:
print(docs[0].page_content[:500])
docs[0].metadata

2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN


{'producer': 'Aspose.Words for Java 24.2.0',
 'creator': 'Aspose.Words',
 'creationdate': '',
 'author': 'e-Parliament@europarl.europa.eu',
 'dmxml.render.id': '652525',
 'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa',
 'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml',
 'source': 'data/CRE-10-2026-10-05_EN.pdf',
 'total_pages': 113}

## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [19]:
# Save your question into a variable called `query`
query = "Summarize the debate on counterfeit medicines sold online."

# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`
retrieved_docs = vector_store.similarity_search(query, k=4)

for doc in retrieved_docs:
    print(doc.page_content[:300], "\n---")

must include safety features on their packaging, like a unique identifier with a product code, serial
58  05-10-2026 
number, batch number and expiry date. They should also include an anti-tampering device so any 
interference with outer packaging is visible. These features are backed up by a singl 
---
become victims. 
 
We have the laws, but they are not respected. Platforms are obliged to remove illegal ads. By law, 
they are not allowed to profit from fake and counterfeit medicine. At our borders, our customs 
authorities need the staff, tools and cooperation to stop illegal medicines before th 
---
augančiai nelegalių ir falsifikuotų vaistų prekybos internete problemai. Tai nėra vien įprasto 
finansinio sukčiavimo klausimas, kai žmonės apgaunami sumokėti už prekes, kurių taip ir 
negauna. Vaisto atveju pasekmės gali būti daug rimtesnės nei vien tik finansiniai nuostoliai. Jos 
gali tiesiogiai  
---
comprometidos, surgen otros modelos que comprometen la seguridad y acaban con la c

Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [20]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Sonra temel bir istem oluşturun:

In [21]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [22]:
response = llm.invoke(prompt)
Markdown(response.text)

Based on the provided text, here is a summary of the debate regarding counterfeit and illegal medicines sold online:

### 1. The Scale and Danger of the Threat
* **Severe Health Risks:** Fake medicines sold online are not just financial scams; they directly endanger public health and cost lives. Counterfeit products may contain no active ingredients, the wrong doses, or dangerous toxic substances such as rat poison, wall paint, or cement.
* **Targeted Products:** E-commerce platforms and illicit networks frequently promote and sell dangerous products, including unauthorized prescription drugs, fake vaccines, weight-loss treatments, and diabetes medications. Even legitimate drugs sold outside the legal supply chain can become dangerous due to improper storage and transport.
* **Exploitation of Digital Spaces:** Criminal networks increasingly use online marketplaces, social media, and digital ads (on platforms like Google, TikTok, Meta, Temu, and AliExpress) to promote and profit from illegal medical products. 

### 2. Regulatory and Enforcement Failures
* **Laws Exist, but Compliance Lacks:** While the EU and member states have strict laws and safety features (such as unique identifiers, serial numbers, anti-tampering devices, and the common EU pharmacy logo), these rules are frequently ignored or inadequately enforced. 
* **Corporate Accountability:** Tech platforms and online marketplaces are legally obligated to remove illegal ads and are barred from profiting from fake medicines, yet many continue to facilitate their sale. Speakers emphasized that the European Commission must follow through with strict enforcement and heavy fines against non-compliant platforms.

### 3. Proposed Solutions and Actions
* **Stronger Enforcement & Fines:** The EU must use existing tools—such as the Digital Services Act—to hold online marketplaces accountable, backed by heavy financial penalties (referencing recent fines against Temu and AliExpress) and investigations by the European Anti-Fraud Office (OLAF).
* **Cross-Sector Cooperation:** There is an urgent call for better, faster real-time information sharing and operational cooperation among online platforms, national medicine regulatory agencies, customs authorities, and law enforcement to intercept illegal shipments before they reach consumers.
* **Criminalizing Counterfeiting:** Speakers highlighted the need to treat medical counterfeiting strictly as a severe threat to public safety rather than a mere intellectual property crime, suggesting that violations should face criminal prosecution.
* **Border Controls:** Customs authorities require adequate staffing, tools, and cross-border cooperation to stop illegal medicines at EU borders.
* **Public Awareness and Professional Trust:** Citizens must be educated not to trust social media ads for healthcare. Instead, they should rely on authorized channels, professional healthcare advice (doctors and pharmacists), and trusted local distribution networks—such as physical pharmacies—to ensure secure access to genuine treatments.

Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [23]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [24]:
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [25]:
the_prompt = prompt_template.invoke({"context": docs_content, "question": query})

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [26]:
response = llm.invoke(the_prompt)
Markdown(response.text)

The debate on counterfeit medicines sold online highlights that falsified drugs—which may contain toxic substances or lack active ingredients—pose severe risks to public health and consumer safety. While digital marketplaces and legislation exist to regulate sales and hold platforms accountable, critics argue that enforcement remains insufficient and technological platforms continue to profit from illegal ads. Speakers stress the need for stronger cooperation among customs, law enforcement, and online platforms to remove illegal offers and protect citizens from dangerous scams.

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [27]:
from langchain_chroma import Chroma

chroma_store = Chroma(
    collection_name="ep_debates",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [28]:
def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    # Load the PDF file
    loader = PyPDFLoader(file_path, mode="single")
    pdf = loader.load()

    # Split the pages into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    all_splits = text_splitter.split_documents(pdf)

    # # Add the session_date to the metadata
    # for split in all_splits:
    #     split.metadata['session_date'] = session_date

    # Add the chunks to the vector store (in batches, to respect the free-tier quota)
    document_ids = add_in_batches(vector_store, all_splits)

    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [29]:
# Skip if this file is already in the persistent store, so a rerun doesn't spend quota again
if chroma_store._collection.count() == 0:
    document_ids = embed_and_store(file_path, chroma_store)
print(chroma_store._collection.count(), "chunks in Chroma")

10/207 chunks embedded
20/207 chunks embedded
30/207 chunks embedded
40/207 chunks embedded
50/207 chunks embedded
60/207 chunks embedded
70/207 chunks embedded
80/207 chunks embedded
90/207 chunks embedded
100/207 chunks embedded
110/207 chunks embedded
120/207 chunks embedded
130/207 chunks embedded
140/207 chunks embedded
150/207 chunks embedded
160/207 chunks embedded
170/207 chunks embedded
180/207 chunks embedded
190/207 chunks embedded
200/207 chunks embedded
207/207 chunks embedded
207 chunks in Chroma


#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [30]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.text

👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [31]:
Markdown(answer("What safety features must medicine packaging have according to the debate?", chroma_store, llm))

According to the debate, prescription medicine packaging must include safety features such as a unique identifier containing a product code, serial number, batch number, and expiry date. Additionally, packages should feature an anti-tampering device to make any interference with the outer packaging visible.

🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [32]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store.
    Session_date is added to the metadata of each chunk."""
    loader = PyPDFLoader(file_path, mode="single")
    pdf = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    all_splits = text_splitter.split_documents(pdf)

    # Add metadata BEFORE storing, so it's saved alongside each embedding
    for split in all_splits:
        split.metadata["session_date"] = session_date

    document_ids = add_in_batches(vector_store, all_splits)

    return document_ids

👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [33]:
# Separate collection, so the plain one from before stays intact
chroma_dated = Chroma(
    collection_name="ep_debates_dated",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

if chroma_dated._collection.count() == 0:
    embed_and_store_fancy(file_path, chroma_dated, session_date="2026-10-05")

print(chroma_dated._collection.count(), "chunks in dated collection")
chroma_dated.get(limit=2, include=["metadatas"])["metadatas"]

10/207 chunks embedded
20/207 chunks embedded
30/207 chunks embedded
40/207 chunks embedded
50/207 chunks embedded
60/207 chunks embedded
70/207 chunks embedded
80/207 chunks embedded
90/207 chunks embedded
100/207 chunks embedded
110/207 chunks embedded
120/207 chunks embedded
130/207 chunks embedded
140/207 chunks embedded
150/207 chunks embedded
160/207 chunks embedded
170/207 chunks embedded
180/207 chunks embedded
190/207 chunks embedded
200/207 chunks embedded
207/207 chunks embedded
207 chunks in dated collection


[{'creationdate': '',
  'session_date': '2026-10-05',
  'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa',
  'author': 'e-Parliament@europarl.europa.eu',
  'producer': 'Aspose.Words for Java 24.2.0',
  'total_pages': 113,
  'source': 'data/CRE-10-2026-10-05_EN.pdf',
  'dmxml.render.id': '652525',
  'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml',
  'creator': 'Aspose.Words'},
 {'total_pages': 113,
  'dmxml.render.id': '652525',
  'creationdate': '',
  'session_date': '2026-10-05',
  'producer': 'Aspose.Words for Java 24.2.0',
  'creator': 'Aspose.Words',
  'uid': 'eu.europa.europarl-DIN1-2026-0000279821_01.00-xm-01.00_text-xml',
  'dmxml.render.traceid': '6accf713b68a1565c86ebc26d7bc8aaa',
  'source': 'data/CRE-10-2026-10-05_EN.pdf',
  'author': 'e-Parliament@europarl.europa.eu'}]

Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [34]:
def answer_fancy(query, vector_store, llm, session_date=None, prompt_template=None):
    """Answer a query using only documents from the given session date (if provided)."""
    # Filter on metadata during retrieval
    search_filter = {"session_date": session_date} if session_date else None
    retrieved_docs = vector_store.similarity_search(query, k=6, filter=search_filter)

    if not retrieved_docs:
        return f"No documents found for session date {session_date}."

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke({"context": docs_content, "question": query})
    response = llm.invoke(prompt)

    return response.text

In [35]:
question = "What did MEPs say about the humanitarian situation in Ukraine?"

print("== 2026-10-05")
display(Markdown(answer_fancy(question, chroma_dated, llm, session_date="2026-10-05")))

print("== 2026-10-06")
print(answer_fancy(question, chroma_dated, llm, session_date="2026-10-06"))

== 2026-10-05


Based on the provided context, MEPs described the humanitarian situation in Ukraine as disastrous, tragic, and nearing a tipping point, with thousands of civilians killed and millions requiring humanitarian aid. They highlighted that Russian attacks and occupations have left people in places like Oleshky trapped without food, water, medicine, or heating, forcing some to resort to eating grass or tree bark. Furthermore, MEPs noted that Russia is using energy and winter as weapons against civilians to break Ukraine's resolve.

== 2026-10-06
No documents found for session date 2026-10-06.


Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!